In [ ]:
# libraries
import os
import json
import time
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import nbformat
from IPython.display import display

In [ ]:
# globals; run through run_pipeline.ipynb
ROOT = Path(os.environ['NCA_PROJECT_ROOT'])
TASK = ROOT / 'f_phase2_train'
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CFG = json.loads((INPUT / 'config.json').read_text())
DEVICE = torch.device(CFG['device'])
torch.set_num_threads(CFG['cpu_threads'])
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

In [ ]:
# functions passed through c_models/output; definitions only, no training side effects
runtime_nb = nbformat.read(INPUT / 'runtime.ipynb', as_version=4)
for runtime_cell in runtime_nb.cells:
    exec(compile(runtime_cell.source, 'runtime.ipynb', 'exec'), globals())
PACK = torch.load(INPUT / 'dataset.pt', weights_only=True, map_location='cpu')
SPECS = CFG['models']

In [ ]:
# content identities for exact checkpoint resumption
data_hash = hashlib.sha256((INPUT / 'dataset.pt').read_bytes()).hexdigest()
code_hash = (INPUT / 'runtime.sha256').read_text()
run_rows = []

In [ ]:
# Phase 1 best checkpoints are initialization only; Phase 2 gets its own optimizer and selection
phase1_manifest = pd.read_csv(INPUT / 'phase1_manifest.csv')

In [ ]:
# train Phase 2; identical condition/augmentation schedules within each paired replicate
runs = OUTPUT / 'phase2_runs'
runs.mkdir(exist_ok=True)
for regime in CFG['budget_regimes']:
    for replicate in range(CFG['replicates']):
        for spec in SPECS:
            name = spec['name']
            run_name = f'{regime}__{name}__r{replicate}'
            phase1_row = phase1_manifest.query('model == @name and regime == @regime and replicate == @replicate').iloc[0]
            warm_start = INPUT / 'phase1_runs' / phase1_row['checkpoint'] if CFG['phase2_initialization'] == 'phase1' else None
            result = fit_run(spec, CFG, PACK, 2, regime, replicate, runs / run_name,
                             data_hash, code_hash, warm_start=warm_start)
            run_rows.append(result)
            pd.DataFrame(run_rows).to_csv(OUTPUT / 'phase2_manifest.csv', index=False)
display(pd.DataFrame(run_rows))